In [1]:
# ======================================================================================
# ALS + ItemCF + LightGBM Reranking - WITH HISTORY (Recommend already purchased items)
# ======================================================================================
!pip install implicit lightgbm

import gc
import pickle
import numpy as np
import pandas as pd
import polars as pl
from datetime import datetime
from scipy.sparse import csr_matrix
from scipy import sparse
import implicit
import lightgbm as lgb
from collections import defaultdict
import json

# Cấu hình
pl.Config.set_tbl_rows(20)
RANDOM_STATE = 42
USE_GPU = True

# ======================================================================================
# 1. LOAD DATA
# ======================================================================================
print("=" * 60)
print("🚀 ALS + ItemCF + LightGBM Reranking (WITH HISTORY)")
print("=" * 60)

print("\n>>> [1/9] Loading Data...")
data_path = '/kaggle/input/dataset'

# Load Ground Truth
gt_path = f'{data_path}/final_groundtruth.pkl'
with open(gt_path, 'rb') as f:
    gt_data = pickle.load(f)

ground_truth_dict = {}
if isinstance(gt_data, pd.DataFrame):
    gt_data['customer_id'] = gt_data['customer_id'].astype(str)
    first_item = gt_data['item_id'].iloc[0]
    if isinstance(first_item, (list, np.ndarray)):
        ground_truth_dict = dict(zip(gt_data['customer_id'], gt_data['item_id']))
    else:
        gt_data['item_id'] = gt_data['item_id'].astype(str)
        ground_truth_dict = gt_data.groupby('customer_id')['item_id'].apply(list).to_dict()
else:
    ground_truth_dict = {str(k): [str(i) for i in (v if isinstance(v, list) else [v])] 
                         for k, v in gt_data.items()}

print(f"   Total GT users: {len(ground_truth_dict):,}")

# Load Transactions
df_trans = pl.read_parquet(f'{data_path}/recommendation dataset/sales_pers.purchase_history_daily_chunk_*.parquet')
df_trans = df_trans.rename({
    df_trans.columns[0]: 'timestamp', 
    df_trans.columns[1]: 'user_id', 
    df_trans.columns[2]: 'item_id'
}).select([
    pl.col('user_id').cast(pl.String), 
    pl.col('item_id').cast(pl.String), 
    (pl.col('timestamp') * 1000).cast(pl.Datetime("ms")).alias('timestamp')
])

print(f"   Transactions from parquet: {len(df_trans):,}")

# Load additional data from 01-2025.pkl
jan_2025_path = f'{data_path}/01-2025.pkl'
try:
    with open(jan_2025_path, 'rb') as f:
        jan_2025_data = pickle.load(f)
    
    if isinstance(jan_2025_data, pd.DataFrame):
        jan_df = pl.from_pandas(jan_2025_data)
    elif isinstance(jan_2025_data, dict):
        rows = []
        for cid, items in jan_2025_data.items():
            if isinstance(items, list):
                for item in items:
                    rows.append({'user_id': str(cid), 'item_id': str(item), 'timestamp': datetime(2025, 1, 15)})
            else:
                rows.append({'user_id': str(cid), 'item_id': str(items), 'timestamp': datetime(2025, 1, 15)})
        jan_df = pl.DataFrame(rows)
    else:
        jan_df = None
        print(f"   ⚠️ Unknown format for 01-2025.pkl: {type(jan_2025_data)}")
    
    if jan_df is not None:
        # Ensure columns match
        if 'timestamp' not in jan_df.columns:
            jan_df = jan_df.with_columns(pl.lit(datetime(2025, 1, 15)).alias('timestamp'))
        
        # Convert timestamp if it's Int64 (unix timestamp)
        if jan_df['timestamp'].dtype == pl.Int64:
            jan_df = jan_df.with_columns(
                (pl.col('timestamp') * 1000).cast(pl.Datetime("ms")).alias('timestamp')
            )
        
        jan_df = jan_df.select([
            pl.col('user_id').cast(pl.String),
            pl.col('item_id').cast(pl.String),
            pl.col('timestamp')
        ])
        
        df_trans = pl.concat([df_trans, jan_df])
        print(f"   ✅ Added Jan 2025 data: {len(jan_df):,} transactions")
        print(f"   Total transactions: {len(df_trans):,}")
        
except FileNotFoundError:
    print(f"   ⚠️ File not found: {jan_2025_path}")
except Exception as e:
    print(f"   ⚠️ Error loading 01-2025.pkl: {e}")

# Load User Mapping
df_user = pl.read_parquet(f'{data_path}/recommendation dataset/sales_pers.user_chunk_*.parquet')
df_user = df_user.rename({
    df_user.columns[0]: 'customer_id',
    df_user.columns[-2]: 'user_id'
}).select([
    pl.col('user_id').cast(pl.String), 
    pl.col('customer_id').cast(pl.String)
]).unique(subset=['user_id'])

pdf_user = df_user.to_pandas()
cust_to_user = dict(zip(pdf_user['customer_id'], pdf_user['user_id']))
user_to_cust = dict(zip(pdf_user['user_id'], pdf_user['customer_id']))

# ======================================================================================
# 2. BUILD POPULARITY, HISTORY, FREQUENCY & RECENCY
# ======================================================================================
print("\n>>> [2/9] Building Popularity, History, Frequency & Recency...")

reference_date = datetime(2024, 12, 31)

# Popularity
pop_df = df_trans.filter(pl.col("timestamp") >= datetime(2024, 10, 1))
if pop_df.height < 1000: pop_df = df_trans
item_popularity = {}
for row in pop_df.group_by('item_id').len().iter_rows():
    item_popularity[str(row[0])] = row[1]
    
top_items_df = pop_df.group_by('item_id').len().sort('len', descending=True).head(100)
global_top_items = top_items_df['item_id'].to_list()

# User history
user_history_dict = {}
history_df = df_trans.filter(
    (pl.col("timestamp") >= datetime(2024, 1, 1)) & 
    (pl.col("timestamp") <= datetime(2025, 1, 31))
).group_by('user_id').agg(pl.col('item_id').alias('items'))

for row in history_df.iter_rows():
    user_id, items = row[0], row[1]
    user_history_dict[user_id] = list(set(items))

customer_history_dict = {}
for uid, items in user_history_dict.items():
    cid = user_to_cust.get(uid)
    if cid:
        customer_history_dict[cid] = items

user_purchase_count = {}
for uid, items in user_history_dict.items():
    user_purchase_count[uid] = len(items)

print(f"   Users with history: {len(user_history_dict):,}")
print(f"   Items with popularity: {len(item_popularity):,}")

# ===== BUILD FREQUENCY & RECENCY DATA =====
print("   Building frequency & recency data...")

user_item_frequency = defaultdict(lambda: defaultdict(int))
user_item_last_purchase = defaultdict(lambda: defaultdict(lambda: datetime(2024, 1, 1)))
user_item_first_purchase = defaultdict(lambda: defaultdict(lambda: datetime(2024, 12, 31)))

train_trans = df_trans.filter(
    (pl.col("timestamp") >= datetime(2024, 1, 1)) & 
    (pl.col("timestamp") <= datetime(2025, 1, 31))
)

for row in train_trans.iter_rows():
    ts, uid, item_id = row[2], row[0], row[1]
    user_item_frequency[uid][item_id] += 1
    if ts > user_item_last_purchase[uid][item_id]:
        user_item_last_purchase[uid][item_id] = ts
    if ts < user_item_first_purchase[uid][item_id]:
        user_item_first_purchase[uid][item_id] = ts

customer_item_frequency = defaultdict(lambda: defaultdict(int))
customer_item_last_purchase = defaultdict(lambda: defaultdict(lambda: datetime(2024, 1, 1)))
customer_item_first_purchase = defaultdict(lambda: defaultdict(lambda: datetime(2024, 12, 31)))

for uid, item_freq in user_item_frequency.items():
    cid = user_to_cust.get(uid)
    if cid:
        for item_id, freq in item_freq.items():
            customer_item_frequency[cid][item_id] = freq
            customer_item_last_purchase[cid][item_id] = user_item_last_purchase[uid][item_id]
            customer_item_first_purchase[cid][item_id] = user_item_first_purchase[uid][item_id]

# Item repurchase rate
item_buyers = defaultdict(set)
item_repeat_buyers = defaultdict(set)
for uid, item_freq in user_item_frequency.items():
    for item_id, freq in item_freq.items():
        item_buyers[item_id].add(uid)
        if freq > 1:
            item_repeat_buyers[item_id].add(uid)

item_repurchase_rate = {}
for item_id in item_buyers:
    n_buyers = len(item_buyers[item_id])
    n_repeat = len(item_repeat_buyers.get(item_id, set()))
    item_repurchase_rate[item_id] = n_repeat / n_buyers if n_buyers > 0 else 0

print(f"   Users with frequency data: {len(customer_item_frequency):,}")
print(f"   Items with repurchase rate: {len(item_repurchase_rate):,}")

# Eval history (for reference only, we don't filter)
hist_for_eval = {}
for cid in ground_truth_dict.keys():
    uid = cust_to_user.get(cid)
    if uid and uid in user_history_dict:
        hist_for_eval[cid] = [str(i) for i in user_history_dict[uid]]
    elif cid in customer_history_dict:
        hist_for_eval[cid] = [str(i) for i in customer_history_dict[cid]]
    else:
        hist_for_eval[cid] = []

gt_str = {k: [str(i) for i in v] for k, v in ground_truth_dict.items()}

# ======================================================================================
# 3. BUILD MATRIX & TRAIN ALS
# ======================================================================================
print("\n>>> [3/9] Training ALS...")

train_df = df_trans.filter(
    (pl.col("timestamp") >= datetime(2024, 1, 1)) & 
    (pl.col("timestamp") <= datetime(2025, 1, 31))
)

print(f"   Training data: {len(train_df):,} transactions")

unique_users = train_df['user_id'].unique().to_list()
unique_items = train_df['item_id'].unique().to_list()

user_to_idx = {v: k for k, v in enumerate(unique_users)}
idx_to_item = {k: v for k, v in enumerate(unique_items)}
item_to_idx = {v: k for k, v in enumerate(unique_items)}

train_df = train_df.with_columns([
    pl.col("user_id").replace_strict(user_to_idx, default=None).cast(pl.Int32).alias("u_idx"),
    pl.col("item_id").replace_strict(item_to_idx, default=None).cast(pl.Int32).alias("i_idx")
]).drop_nulls()

counts = train_df.group_by(['u_idx', 'i_idx']).len()
rows = counts['u_idx'].to_numpy()
cols = counts['i_idx'].to_numpy()
data = counts['len'].to_numpy().astype(np.float32)

matrix = csr_matrix((data, (rows, cols)), shape=(len(unique_users), len(unique_items)))

model_als = implicit.als.AlternatingLeastSquares(
    factors=128, regularization=0.1, alpha=40, iterations=15,
    random_state=RANDOM_STATE, use_gpu=USE_GPU
)
model_als.fit(matrix)
print("   ✅ ALS trained")

# ======================================================================================
# 4. COMPUTE ITEM SIMILARITY
# ======================================================================================
print("\n>>> [4/9] Computing Item Similarity...")

item_matrix = matrix.T.tocsr()
norms = sparse.linalg.norm(item_matrix, axis=1)
norms[norms == 0] = 1
item_matrix_normalized = item_matrix.multiply(1.0 / norms.reshape(-1, 1)).tocsr()

n_items = item_matrix.shape[0]
ITEM_SIM_TOP_K = 30
item_similarity = {}

batch_size = 2000
for start in range(0, n_items, batch_size):
    end = min(start + batch_size, n_items)
    batch_sim = (item_matrix_normalized[start:end] @ item_matrix_normalized.T).toarray()
    for i, row in enumerate(batch_sim):
        item_idx = start + i
        row[item_idx] = 0
        top_indices = np.argsort(row)[::-1][:ITEM_SIM_TOP_K]
        item_similarity[item_idx] = [(int(j), float(row[j])) for j in top_indices if row[j] > 0]

print(f"   ✅ Item similarity computed ({len(item_similarity):,} items)")

# ======================================================================================
# 5. BUILD LTR TRAINING DATA (NO HISTORY FILTERING!)
# ======================================================================================
print("\n>>> [5/9] Building LTR Training Data (WITH HISTORY)...")

import random
sample_users = random.sample(list(customer_history_dict.keys()), min(30000, len(customer_history_dict)))

user_items_csr = matrix.tocsr()
ALS_WEIGHT = 1.0
ITEMCF_WEIGHT = 0.5

ltr_rows = []
ltr_labels = []
ltr_groups = []

for cid in sample_users:
    uid = cust_to_user.get(cid)
    u_idx = user_to_idx.get(uid) if uid else None
    
    if u_idx is None:
        continue
    
    gt_items = set(str(i) for i in ground_truth_dict.get(cid, []))
    
    # Get candidates - NO HISTORY FILTERING
    item_scores = {}
    
    # ALS - filter_already_liked_items=FALSE -> Recommend already bought items
    try:
        als_ids, als_scores = model_als.recommend([u_idx], user_items_csr[[u_idx]], N=50, filter_already_liked_items=False)
        for item_idx, score in zip(als_ids[0], als_scores[0]):
            item_id = idx_to_item[item_idx]
            # NO filter hist_items
            item_scores[item_id] = {'als_score': float(score), 'itemcf_score': 0.0}
    except:
        continue
    
    # ItemCF - NO HISTORY FILTERING
    user_items = user_items_csr[u_idx].indices[:30]
    for bought_item_idx in user_items:
        if bought_item_idx in item_similarity:
            for sim_idx, sim_score in item_similarity[bought_item_idx][:15]:
                item_id = idx_to_item.get(sim_idx)
                if item_id:  # NO filter hist_items
                    if item_id not in item_scores:
                        item_scores[item_id] = {'als_score': 0.0, 'itemcf_score': 0.0}
                    item_scores[item_id]['itemcf_score'] += sim_score
    
    if len(item_scores) < 10:
        continue
    
    group_size = 0
    user_n_purchases = user_purchase_count.get(uid, 0)
    
    for item_id, scores in list(item_scores.items())[:100]:
        als_score = scores['als_score']
        itemcf_score = scores['itemcf_score']
        
        label = 1 if item_id in gt_items else 0
        
        popularity = item_popularity.get(item_id, 0)
        user_item_freq = customer_item_frequency[cid].get(item_id, 0)
        
        if user_item_freq > 0:
            last_purchase = customer_item_last_purchase[cid][item_id]
            first_purchase = customer_item_first_purchase[cid][item_id]
            days_since_last = max(0, (reference_date - last_purchase).days)
            days_since_first = max(0, (reference_date - first_purchase).days)
            recency_score = 1.0 / (1.0 + max(1, days_since_last) / 30.0)
        else:
            days_since_last = 365
            days_since_first = 365
            recency_score = 0.0
        
        repurchase_rate = item_repurchase_rate.get(item_id, 0)
        
        features = [
            als_score,
            itemcf_score,
            als_score * ALS_WEIGHT + itemcf_score * ITEMCF_WEIGHT,
            popularity,
            np.log1p(popularity),
            user_n_purchases,
            np.log1p(user_n_purchases),
            1 if als_score > 0 else 0,
            1 if itemcf_score > 0 else 0,
            user_item_freq,
            np.log1p(user_item_freq),
            1 if user_item_freq > 0 else 0,
            1 if user_item_freq > 2 else 0,
            recency_score,
            days_since_last,
            np.log1p(days_since_last),
            days_since_first,
            repurchase_rate,
        ]
        
        ltr_rows.append(features)
        ltr_labels.append(label)
        group_size += 1
    
    if group_size > 0:
        ltr_groups.append(group_size)

X_train = np.array(ltr_rows)
y_train = np.array(ltr_labels)

print(f"   Training samples: {len(X_train):,}")
print(f"   Training groups: {len(ltr_groups):,}")
print(f"   Positive rate: {y_train.mean()*100:.2f}%")

# ======================================================================================
# 6. TRAIN LIGHTGBM RANKER
# ======================================================================================
print("\n>>> [6/9] Training LightGBM Ranker...")

train_data = lgb.Dataset(X_train, label=y_train, group=ltr_groups)

lgb_params = {
    'objective': 'lambdarank',
    'metric': 'ndcg',
    'ndcg_at': [10],
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,
    'device': 'gpu',
    'gpu_platform_id': 0,
    'gpu_device_id': 0,
    'seed': RANDOM_STATE,
}

model_lgb = lgb.train(
    lgb_params,
    train_data,
    num_boost_round=200,
)

print(f"   ✅ LightGBM trained ({model_lgb.num_trees()} trees)")

# ======================================================================================
# 7. GENERATE PREDICTIONS WITH RERANKING (NO HISTORY FILTERING!)
# ======================================================================================
print("\n>>> [7/9] Generating Predictions with Reranking (WITH HISTORY)...")

gt_customer_ids = list(ground_truth_dict.keys())
all_predictions = {}
BATCH_SIZE = 5000

for i in range(0, len(gt_customer_ids), BATCH_SIZE):
    if i % 20000 == 0: print(f"   Processing batch {i}...")
    batch_cids = gt_customer_ids[i : i + BATCH_SIZE]
    
    for cid in batch_cids:
        uid = cust_to_user.get(cid)
        if uid is None and cid in user_to_idx: 
            uid = cid
        
        u_idx = user_to_idx.get(uid) if uid else None
        
        if u_idx is None:
            # Cold start - NO filter
            all_predictions[cid] = global_top_items[:10]
            continue
        
        item_scores = {}
        
        # ALS - filter_already_liked_items=FALSE
        try:
            als_ids, als_scores = model_als.recommend([u_idx], user_items_csr[[u_idx]], N=50, filter_already_liked_items=False)
            for item_idx, score in zip(als_ids[0], als_scores[0]):
                item_id = idx_to_item[item_idx]
                item_scores[item_id] = {'als_score': float(score), 'itemcf_score': 0.0}
        except:
            all_predictions[cid] = global_top_items[:10]
            continue
        
        # ItemCF - NO filter
        user_items = user_items_csr[u_idx].indices[:30]
        for bought_item_idx in user_items:
            if bought_item_idx in item_similarity:
                for sim_idx, sim_score in item_similarity[bought_item_idx][:15]:
                    item_id = idx_to_item.get(sim_idx)
                    if item_id:
                        if item_id not in item_scores:
                            item_scores[item_id] = {'als_score': 0.0, 'itemcf_score': 0.0}
                        item_scores[item_id]['itemcf_score'] += sim_score
        
        if len(item_scores) < 10:
            # Fallback - NO filter
            sorted_items = sorted(item_scores.keys(), key=lambda x: -(item_scores[x]['als_score'] + item_scores[x]['itemcf_score']))
            rec_items = sorted_items[:10]
            for pop in global_top_items:
                if len(rec_items) >= 10:
                    break
                if pop not in rec_items:
                    rec_items.append(pop)
            all_predictions[cid] = rec_items[:10]
            continue
        
        user_n_purchases = user_purchase_count.get(uid, 0)
        candidates = list(item_scores.keys())[:100]
        
        features_batch = []
        for item_id in candidates:
            scores = item_scores[item_id]
            als_score = scores['als_score']
            itemcf_score = scores['itemcf_score']
            popularity = item_popularity.get(item_id, 0)
            
            user_item_freq = customer_item_frequency.get(cid, {}).get(item_id, 0)
            
            if user_item_freq > 0:
                last_purchase = customer_item_last_purchase[cid][item_id]
                first_purchase = customer_item_first_purchase[cid][item_id]
                days_since_last = max(0, (reference_date - last_purchase).days)
                days_since_first = max(0, (reference_date - first_purchase).days)
                recency_score = 1.0 / (1.0 + max(1, days_since_last) / 30.0)
            else:
                days_since_last = 365
                days_since_first = 365
                recency_score = 0.0
            
            repurchase_rate = item_repurchase_rate.get(item_id, 0)
            
            features = [
                als_score,
                itemcf_score,
                als_score * ALS_WEIGHT + itemcf_score * ITEMCF_WEIGHT,
                popularity,
                np.log1p(popularity),
                user_n_purchases,
                np.log1p(user_n_purchases),
                1 if als_score > 0 else 0,
                1 if itemcf_score > 0 else 0,
                user_item_freq,
                np.log1p(user_item_freq),
                1 if user_item_freq > 0 else 0,
                1 if user_item_freq > 2 else 0,
                recency_score,
                days_since_last,
                np.log1p(days_since_last),
                days_since_first,
                repurchase_rate,
            ]
            features_batch.append(features)
        
        lgb_scores = model_lgb.predict(np.array(features_batch))
        sorted_indices = np.argsort(lgb_scores)[::-1]
        rec_items = [candidates[idx] for idx in sorted_indices[:10]]
        
        all_predictions[cid] = rec_items

print(f"   Total predictions: {len(all_predictions):,}")

# ======================================================================================
# 8. EVALUATION
# ======================================================================================
print("\n>>> [8/9] Calculating Score...")

def precision_at_k(pred, gt, hist, filter_bought_items=True, K=10):
    precisions = []
    for user, gt_items_list in gt.items():
        if user not in pred:
            continue
        relevant_items = set(gt_items_list)
        if filter_bought_items and user in hist:
            relevant_items -= set(hist[user])
        if len(relevant_items) == 0:
            continue
        pred_items = pred[user][:K]
        hits = len(set(pred_items) & relevant_items)
        precisions.append(hits / K)
    return np.mean(precisions) if precisions else 0.0

pred_str = {k: [str(i) for i in v] for k, v in all_predictions.items()}

# Score with filter_bought_items=FALSE (allow history)
score = precision_at_k(pred_str, gt_str, hist_for_eval, filter_bought_items=False, K=10)

print(f"\n{'='*60}")
print(f"FINAL SCORE (PRECISION@10 - WITH HISTORY): {score:.6f} ({score*100:.2f}%)")
print(f"{'='*60}")

# ======================================================================================
# 9. EXPORT PREDICTIONS TO JSON
# ======================================================================================
print("\n>>> [9/9] Exporting Predictions to JSON...")

output_path = "predictions_with_hist.json"
with open(output_path, 'w') as f:
    json.dump(pred_str, f, indent=2)

print(f"   ✅ Predictions saved to: {output_path}")
print(f"   Total customers: {len(pred_str):,}")


🚀 ALS + ItemCF + LightGBM Reranking (WITH HISTORY)

>>> [1/9] Loading Data...


/tmp/ipykernel_1423/2522342819.py:37: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  gt_data = pickle.load(f)


   Total GT users: 644,970
   Transactions from parquet: 35,729,825


/tmp/ipykernel_1423/2522342819.py:72: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  jan_2025_data = pickle.load(f)


   ✅ Added Jan 2025 data: 3,298,252 transactions
   Total transactions: 39,028,077

>>> [2/9] Building Popularity, History, Frequency & Recency...
   Users with history: 2,569,978
   Items with popularity: 16,572
   Building frequency & recency data...
   Users with frequency data: 2,569,978
   Items with repurchase rate: 21,095

>>> [3/9] Training ALS...
   Training data: 39,028,077 transactions


  0%|          | 0/15 [00:00<?, ?it/s]

   ✅ ALS trained

>>> [4/9] Computing Item Similarity...
   ✅ Item similarity computed (21,095 items)

>>> [5/9] Building LTR Training Data (WITH HISTORY)...
   Training samples: 2,394,066
   Training groups: 30,000
   Positive rate: 0.46%

>>> [6/9] Training LightGBM Ranker...


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


   ✅ LightGBM trained (200 trees)

>>> [7/9] Generating Predictions with Reranking (WITH HISTORY)...
   Processing batch 0...
   Processing batch 20000...
   Processing batch 40000...
   Processing batch 60000...
   Processing batch 80000...
   Processing batch 100000...
   Processing batch 120000...
   Processing batch 140000...
   Processing batch 160000...
   Processing batch 180000...
   Processing batch 200000...
   Processing batch 220000...
   Processing batch 240000...
   Processing batch 260000...
   Processing batch 280000...
   Processing batch 300000...
   Processing batch 320000...
   Processing batch 340000...
   Processing batch 360000...
   Processing batch 380000...
   Processing batch 400000...
   Processing batch 420000...
   Processing batch 440000...
   Processing batch 460000...
   Processing batch 480000...
   Processing batch 500000...
   Processing batch 520000...
   Processing batch 540000...
   Processing batch 560000...
   Processing batch 580000...
   Proce